In [0]:
%sql

WITH src as 
(
    SELECT 
        gfop.occupancy_id as occupancy_id  ,
        gdpe.person_id as person_id ,
        gfop._last_seen_snapshot as _snapshot_version,
        
        CASE 
            WHEN gfop.tenure_verdict = 'confirmed_current' THEN TRUE 
            WHEN gfop.tenure_verdict = 'confirmed_ended' THEN FALSE
            ELSE NULL
        END As label_is_current,
        coalesce(
            CASE WHEN gfop.start_date_key IS NOT NULL THEN TRUE END ,FALSE
            ) as has_start_date,
        CAST((gfop.start_date_key DIV 10000) AS INT) as start_year,

        (CAST(LEFT(gfop._last_seen_snapshot,4) AS INT)- start_year) as years_since_start,
        gfop.source_count as source_count,
        gfpo.country as post_country,
        gdpe.birth_year_min as person_birth_year,
        gdpe.n_positions as person_n_positions,
        gdpe.is_sanctioned as person_is_sanctioned



        
    FROM pep_dev.gold.fact_occupancy_period AS gfop
    
    LEFT JOIN pep_dev.gold.dim_person AS gdpe
    ON gdpe.person_sk = gfop.person_sk
    LEFT JOIN pep_dev.gold.dim_post AS gfpo 
    ON gfop.post_sk=gfpo.post_sk
    WHERE gfop.person_sk <> -1
    AND gfop.tenure_verdict <> 'derived_ended'

)

-- SELECT * FROM src


INSERT OVERWRITE pep_dev.gold.ml_tenure_features

 (
  occupancy_id,
  person_id, 
  _snapshot_version, 
  label_is_current,
  has_start_date,
  start_year,
  years_since_start, 
  source_count, 
  post_country,
  person_birth_year,
  person_n_positions,
  person_is_sanctioned
  )


SELECT * FROM src;